# 21 — Complete current rules on manually reviewed demonstrations

**DR-65 · Gemini 3.8 Flash and Qwen3.6 · reviewed development cases only**

Does removing rubric history improve tagging while preserving all current rules? Both models receive the same title, permission to use existing knowledge, and either the full original document or its current-rule rendering. Search is off and both arms allow up to two tags. Examples, definitions and reference answers are unchanged.

We use the **54 accepted DR-61 manual reviews**, not all 188 historical records. Three unresolved reviews remain excluded consistently. Previously approved exceptions stay in the answer key with their limitations; higher agreement is not proof that an exception follows the rubric. These are familiar development cases, not an untouched test.


In [ ]:
import copy
import json
import os
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from threading import Event

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path[:0] = [str(root), str(root / "src")]
from dotenv import load_dotenv
from notebooks.tools import verify_sha256, show_table, fingerprint, percent
from notebooks.tools.hosted import chat_request
from notebooks.tools.gem_profile import gem_request
from notebooks.tools.gemini import decode_result
from notebooks.tools.costs import BudgetGuard
from demo_logic.notebook_support import save_private_json
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags

load_dotenv(root / ".env", override=False)
LIVE = os.getenv("NOTEBOOK21_RUN_LIVE", "false").lower() == "true"
folder = root / "reports/rubric-delivery/v1"
reference_path = root / "reports/error-analysis/2026-10-05/approved-reference-v1.json"
verify_sha256(reference_path, "b508d0a3b9b803f7d018816c753e1673860c1a297acd8733607bf6c1f67fe6fe")
reference = json.loads(reference_path.read_text())
reviewed = {r["external_id"]: r for r in reference["records"] if r["score_eligible"]}
ids = sorted(reviewed)
assert len(ids) == 54 and all(r["status"] in {"reviewer_approved", "carried_from_prior_review"} for r in reviewed.values())
audit = json.loads((root / "reports/audit/2026-10-06/calculations.json").read_text())
verify_sha256(root / "reports/background-knowledge/v1/results.json", audit["inputs"]["reports/background-knowledge/v1/results.json"])
prior = json.loads((root / "reports/background-knowledge/v1/results.json").read_text())
config = json.loads((root / "notebooks/prompts/practical-tagging-v1.json").read_text())
models = [c["model"] for c in config["candidates"]]
tags = sorted(topic_tags("v2"))
mapping = json.loads((root / "notebooks/prompts/rubric-v3-operational-v1-map.json").read_text())
for path, sha in [(mapping["source"], mapping["source_sha256"]), (mapping["rendering"], mapping["rendering_sha256"])]:
    verify_sha256(root / path, sha)
raw = (root / mapping["source"]).read_text()
operational = (root / mapping["rendering"]).read_text()
show_table(["Cohort fact", "Value"], [("Accepted manually reviewed cases", len(ids)), ("Unreviewed historical records included", 0), ("New label changes", 0), ("Maximum scheduled calls", 216)])


## 1. Preserve meaning and freeze the comparison

The rendering retains the selection instructions, all twelve definitions and their unchanged examples, all overlap rules, the revised scope decision and all five latest development decisions. It removes approval history and evaluation procedures. Some human-review wording remains because this first test preserves current instructions verbatim.

The full rule mapping is in `notebooks/prompts/rubric-v3-operational-v1-map.json`. We deliberately do not fix the incomplete LGBTQ example in only one arm: that would change more than presentation. Both arms use the identical wrapper and output policy.

Predeclared screening rule: at least +3 percentage points F1; no precision or coverage loss greater than two points; inspect per-tag harm. Passing this diagnostic does not establish the 80/80/95 targets or generalization. The run reserves at most $3 within the existing $50 guard. Costs are conservative estimates, not invoices. The euro dashboard total is not assumed from an ambiguous message.


In [ ]:
payloads = {}
for model in models:
    payloads[model] = {}
    for i in ids:
        base = prior["answers"][model][i]["payload"]
        system = base["messages"][0]["content"] if "messages" in base else base["system_instruction"]
        assert system.endswith(raw)
        wrapper = system[:-len(raw)]
        payloads[model][i] = {}
        for arm, document in [("raw", raw), ("current", operational)]:
            p = copy.deepcopy(base)
            if "messages" in p:
                p["messages"][0]["content"] = wrapper + document
                supplied = json.loads(p["messages"][1]["content"])
            else:
                p["system_instruction"] = wrapper + document
                supplied = json.loads(p["input"])
            assert supplied == {"title": reviewed[i]["title"]} and not p.get("tools")
            payloads[model][i][arm] = p
contract = {"version": "rubric-delivery-v1", "ids": ids, "reference_sha256": fingerprint(reference),
            "mapping": mapping, "reference_changes": False, "search": False, "models": models,
            "output_limit": 2, "manual_reviews_only": True, "new_control_calls": True,
            "gate": {"f1_gain": .03, "maximum_precision_loss": .02, "maximum_coverage_loss": .02},
            "run_limit_usd": 3, "condition_order": "raw first for even sorted index; current first for odd"}
contract["request_fingerprints"] = {m: {i: {a: fingerprint({"endpoint": next(c["endpoint"] for c in config["candidates"] if c["model"] == m), "payload": p}) for a,p in arms.items()} for i,arms in events.items()} for m,events in payloads.items()}
save_private_json(folder / "contract.json", contract)
save_private_json(folder / "reference-limitations.json", [{"external_id": i, "limitation": reviewed[i].get("limitation", "")} for i in ids])


## 2. Run or replay

Default is offline replay. Set `NOTEBOOK21_RUN_LIVE=true` to run missing requests. Fresh controls have their own cache; replay never silently becomes a rerun. The two arms alternate order by event. Up to three independent Gemini events can run at once; Qwen runs sequentially.

Unknown charges, rate limits or infrastructure failures pause that provider. Completed failures remain visible and are not retried automatically. Do not report a paired score until every scheduled answer is accounted for. Existing ledger reservations are retained.


In [ ]:
guard = BudgetGuard(root / "reports/budget/capstone.sqlite", prior_total_microusd=0,
                    prior_research_microusd=0, run_limits={"rubric-delivery-v1": 3_000_000})

def run_model(candidate):
    """Replay exact per-arm responses or run the authorized reviewed cases.

    Alternating arm order reduces ordering effects. A shared stop flag prevents
    further requests after an operational/accounting pause; in-flight calls finish.
    Reference labels are used only by evaluation, never by the request builder.
    """
    model = candidate["model"]
    stopped = Event()
    def run_pair(item):
        index, i = item
        answers = {}
        for arm in (["raw", "current"] if index % 2 == 0 else ["current", "raw"]):
            place = folder / ("qwen" if "Qwen" in model else "gemini") / i / arm
            p = payloads[model][i][arm]
            if "Qwen" in model:
                result = chat_request({**candidate, "key_env": "HETZNER_API_KEY"}, p, tags=tags,
                    cache_dir=place / "cache", allow_live=LIVE and not stopped.is_set(),
                    api_key=os.getenv("HETZNER_API_KEY"), max_retry_wait_seconds=0)
            else:
                failed = sorted((place / "attempts").glob("*-finished.json"))
                if failed:
                    result = json.loads(failed[-1].read_text())
                    assert result["request_sha256"] == fingerprint({"endpoint": candidate["endpoint"], "payload": p})
                    if result["status"] == "ok":
                        result.update(decode_result(result["response"], tags))
                else:
                    result = gem_request(p, event_id=i, folder=place, key=os.getenv("GEMINI_API_KEY"),
                        tags=tags, guard=guard, live=LIVE and not stopped.is_set(), decoder=decode_result,
                        run_id="rubric-delivery-v1", research_budget=False)
            if result.get("status") == "failed" and result.get("error") == "Invalid or incomplete model response":
                result = {**result, "transport_status": "failed", "status": "invalid_response", "predicted": []}
            if result.get("accounting_pending") or result["status"] not in {"ok", "invalid_response"}:
                stopped.set()
            answers[arm] = {k:v for k,v in result.items() if k != "cached"}
        print(model, index + 1, "/54", {a:r["status"] for a,r in answers.items()}, flush=True)
        return i, answers
    with ThreadPoolExecutor(max_workers=1 if "Qwen" in model else 3) as pool:
        return model, dict(pool.map(run_pair, enumerate(ids)))

with ThreadPoolExecutor(max_workers=2) as pool:
    answers = dict(pool.map(run_model, config["candidates"]))
complete = all(r["status"] in {"ok", "invalid_response"} and not r.get("accounting_pending")
               for m in models for pair in answers[m].values() for r in pair.values())
print("Comparison complete:", complete)


## 3. Paired quality and tag-level changes

Precision measures matching suggestions; recall measures reference topics recovered; coverage measures events receiving at least one suggestion. We also show complete tag-set agreement. All measures use the same accepted manual answers. Failed outputs count as empty predictions. These references include approved exceptions; disagreement does not automatically establish model error.


In [ ]:
if not complete:
    raise RuntimeError("Comparison paused or missing responses. Inspect request statuses before reporting paired results.")
metrics = {}
for model in models:
    metrics[model] = {}
    for arm in ["raw", "current"]:
        metrics[model][arm] = multilabel_metrics([reviewed[i]["reference_tags"] for i in ids],
            [answers[model][i][arm].get("predicted", []) for i in ids], tags)
save_private_json(folder / "results.json", {"contract":contract,"answers":answers,"metrics":metrics})
show_table(["Model", "Rubric", "Precision", "Recall", "F1", "Coverage", "Exact match"],
    [(m,a,*[percent(metrics[m][a][k]) for k in ["micro_precision","micro_recall","micro_f1","coverage","exact_match"]])
     for m in models for a in ["raw","current"]])
for m in models:
    before, after = metrics[m]["raw"], metrics[m]["current"]
    numeric_gate = (after["micro_f1"]-before["micro_f1"] >= .03 and
                    after["micro_precision"]-before["micro_precision"] >= -.02 and
                    after["coverage"]-before["coverage"] >= -.02)
    print(m, "passes numeric screening rule:", numeric_gate, "— inspect per-tag changes too")
    show_table(["Tag", "Reference count", "Missing: raw", "Missing: current", "Extra: raw", "Extra: current"],
        [(t,before["per_tag"][t]["support"],before["per_tag"][t]["fn"],after["per_tag"][t]["fn"],
          before["per_tag"][t]["fp"],after["per_tag"][t]["fp"]) for t in tags])


## 4. Decision

Read the paired results before choosing any further changes. An improvement on these familiar reviewed cases is development evidence, not a fresh test. Preserve all approved answers and their limitations. Any later confirmation must also use manually reviewed references; no unreviewed historical labels will enter the new benchmark. New labeling or another experiment is not automatic.


## 5. What we learned

**Removing history did not improve either model on the same 54 reviewed demonstrations.**

- Gemini 3.8 Flash: full document **84.0% precision, 71.6% recall, 77.3% F1, 98.1% coverage**. Current-rule rendering: **78.2%, 69.3%, 73.5%, 96.3%**.
- Qwen3.6: full document **71.0% precision, 75.0% recall, 72.9% F1, 100% coverage**. Current-rule rendering: **69.6%, 72.7%, 71.1%, 100%**.
- Both fail the predeclared improvement screen. All 216 responses were valid, so outages are not the cause.
- Earlier headline scores already used these 54 reviews. This experiment restricts provider calls as well as scoring; the comparison is not improved merely by dropping historical labels.

Keep the full-document Gemini candidate and the earlier Qwen baseline. Do not adopt this rendering as an improvement. First resolve documented rule/reference exceptions before deciding on one further examples-or-context test. No reference answers changed and no new review queue is required to inspect this result.

Read the [public result summary](../docs/results/reviewed-rubric-delivery.md) for timing, conservative cost estimates and limitations.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
measures = ["micro_precision", "micro_recall", "micro_f1", "coverage"]
x = np.arange(len(measures))
for ax, model in zip(axes, models):
    for offset, arm, label in [(-0.18, "raw", "Full document"), (0.18, "current", "Current rules")]:
        ax.bar(x + offset, [100 * metrics[model][arm][k] for k in measures], .34, label=label)
    ax.set_title("Qwen3.6" if "Qwen" in model else "Gemini 3.8 Flash")
    ax.set_xticks(x, ["Precision", "Recall", "F1", "Coverage"])
    ax.set_ylim(0, 105)
axes[0].set_ylabel("Percent")
axes[1].legend()
fig.suptitle("Same 54 manually reviewed cases; fresh paired calls")
fig.tight_layout()
display(fig)
plt.close(fig)
